In [2]:
import pandas as pd
import numpy as np
from get_metadata_genres import get_imdb, get_tmdb, tmdb_genres_map
from wiki import get_movie_synopsis as get_wiki_synopsis
import re

{'title': 'The Matrix', 'release_date': '1999-03-31', 'id': 603, 'overview': 'Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful computers who now rule the earth.', 'genres': [28, 878]}
Iron Man is a superhero appearing in American comic books published by Marvel Comics. Co-created by writer and editor Stan Lee, developed by scripter Larry Lieber, and designed by artists Don Heck and Jack Kirby, the character first appeared in Tales of Suspense #39 in 1962 (cover dated March 1963) and received his own title with Iron Man #1 in 1968. Shortly after his creation, Iron Man became a founding member of the superhero team, the Avengers, alongside Thor, Ant-Man, the Wasp, and the Hulk. Iron Man stories, individually and with the Avengers, have been published consistently since the character's creation.
Iron Man is the superhero persona of Anthony Edward "Tony" Stark, a businessman and engineer who 

In [3]:
df = pd.read_csv("MovieScripts.csv", delimiter=",", quotechar="\"")
df = df.dropna()
df = df.drop(columns=["URL"])
# df.set_index(keys=["Title"], inplace=True)
df.head()

,Title,Script
0,#AMFAD: All My Friends Are Dead (2024),"1\n [static crackling]\n [man] In life,\n they..."
1,#Followme (2019),"(MULTICOM JINGLE)\n (OMINOUS MUSIC)\n Hey, eve..."
2,#Horror (2015),That was incredible.\n How did you get to be\n...
3,#IMomSoHard Live (2019),(upbeat music)\n (audience cheering\n and appl...
4,#MenToo (2023),1\n 'Feminism!'\n 'The trending word on all\n ...


In [4]:
def split_t(title) :
    match = re.search(r'\((\d{4})\)', title)
    if match:
        year = match.group(1)
        title_before_year = title.replace(f"({year})", "").strip()
        return title_before_year, year
    else:
        return None

In [ ]:
# df = df.drop(columns=df.columns)
def update_tmdb_data(row):
    movie = row.Title
    print(f"Processing {movie}", end="\r")
    parts = movie.split()
    # Get the data from IMDB
    try :
        imdb_data = get_imdb(movie)
        if imdb_data :
            row["imdb_id"] = imdb_data["id"]
            if "synopsis" in imdb_data :
                row["synopsis"] = imdb_data["synopsis"][0]
            if "genres" in imdb_data :
                genres = imdb_data["genres"]
                for g in range(len(genres)) :
                    genre = "imdb_" + genres[g].lower()
                    row[genre] = g + 1
    except Exception as e:
        print(f"Could not get data from IMDB for {movie}")
    
    # Get the data from TMDB
    try:
        tmdb_data = get_tmdb(' '.join(parts[:-1]))
        if tmdb_data:
            row['tmdb_id'] = tmdb_data['id']
            if 'genres' in tmdb_data:
                for g, genre_id in enumerate(tmdb_data['genres']):
                    genre = f"tmdb_{tmdb_genres_map[genre_id].lower()}"
                    row[genre] = g + 1
            if 'overview' in tmdb_data:
                row['overview'] = tmdb_data['overview']
    except Exception as e:
        print(f"Could not get data from TMDB for {movie}")

    # Get the data from Wikipedia
    try :
        title, year = split_t(movie)
        if title :
            wiki_data = get_wiki_synopsis(movie)
            if wiki_data:
                row['wiki_synopsis'] = wiki_data
    except Exception as e:
        print(f"Could not get data from Wikipedia for {movie}")
    return row

# df = df[df["Title"] == "10 Things I Hate About You (1999)"].apply(update_tmdb_data, axis=1)

In [4]:
df_genres = pd.read_csv("MovieData.csv", delimiter=",")
df_genres.shape

C:\Users\Pierre\AppData\Local\Temp\ipykernel_7792\3656161739.py:1: DtypeWarning: Columns (28,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df_genres = pd.read_csv("MovieData.csv", delimiter=",")


(38163, 52)

In [6]:
current_chunk = 0
chunk_size = 1200
chunks = np.array_split(df, len(df) // chunk_size)
nbchunks = len(chunks)
nbchunks

e:\clone\nlp\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


31

In [ ]:
import concurrent.futures

# Fonction qui traite un chunk de données
def process_chunk(chunk_data):
    # Appliquer la fonction update_tmdb_data à un chunk
    return chunk_data.apply(update_tmdb_data, axis=1)

# Fonction principale pour traiter tous les chunks
def process_all_chunks(chunks, nbchunks):
    results = []
    
    # Utilisation de concurrent.futures pour paralléliser le traitement des chunks
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(process_chunk, chunks[i]) for i in range(nbchunks)]
        
        # Collecte les résultats de chaque chunk traité
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())
    
    # Combinaison de tous les résultats traités dans un DataFrame final
    final_df = pd.concat(results, ignore_index=True)

    # Écriture dans le fichier en mode append, mais uniquement une fois tous les chunks traités
    final_df.to_csv("MovieDataThreadWiki.csv", mode='w', header=True, index=False)
    print("All chunks processed and saved to 'MovieData.csv'.")

# Appel de la fonction principale pour traiter les chunks
process_all_chunks(chunks, nbchunks)


2025-04-29 19:29:55,734 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Avenging+Force+%281986%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_de

2025-04-29 19:30:00,632 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=American+Exit+%282019%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_def

2025-04-29 19:36:52,975 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Big+Kill+%282018%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_default


2025-04-29 19:37:21,352 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Big+Match+%282014%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_default

2025-04-29 19:42:01,223 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Amityville+in+Space+%282022%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_err

...

2025-04-29 20:14:30,200 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=68+Kill+%282017%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_default
 

2025-04-29 20:17:37,899 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Another+Time+%282018%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_defa

2025-04-29 20:18:05,065 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Another+Year+%282010%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_defa

2025-04-29 20:19:34,278 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Basket+Case+%281982%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_defau

2025-04-29 20:20:18,625 CRITICAL [imdbpy] e:\clone\nlp\.venv\lib\site-packages\imdb\_exceptions.py:32: IMDbDataAccessError exception raised; args: ({'errcode': None, 'errmsg': 'None', 'url': 'https://www.imdb.com/find/?q=Bastille+Day+%282016%29&s=tt', 'proxy': '', 'exception type': 'IOError', 'original exception': <HTTPError 405: 'Not Allowed'>},); kwds: {}
Traceback (most recent call last):
  File "e:\clone\nlp\.venv\lib\site-packages\imdb\parser\http\__init__.py", line 233, in retrieve_unicode
    response = uopener.open(url)
  File "C:\Python39\lib\urllib\request.py", line 523, in open
    response = meth(req, response)
  File "C:\Python39\lib\urllib\request.py", line 632, in http_response
    response = self.parent.error(
  File "C:\Python39\lib\urllib\request.py", line 561, in error
    return self._call_chain(*args)
  File "C:\Python39\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "C:\Python39\lib\urllib\request.py", line 641, in http_error_defa

In [6]:
retry = pd.read_csv("MovieDataRetry.csv", delimiter=",")
retry

C:\Users\Aniss\AppData\Local\Temp\ipykernel_25456\3034288060.py:1: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  retry = pd.read_csv("MovieDataRetry.csv", delimiter=",")


,Script,Title,imdb_action,imdb_adult,imdb_adventure,imdb_animation,imdb_biography,imdb_comedy,imdb_crime,imdb_documentary,...,tmdb_horror,tmdb_id,tmdb_music,tmdb_mystery,tmdb_romance,tmdb_science fiction,tmdb_thriller,tmdb_tv movie,tmdb_war,tmdb_western
0,World War One...\n brought defeat to the three...,1920 Bitwa Warszawska (Battle Of Warsaw 1920) ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,87077.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN
1,Present by CJ Entertainment\n Produced by Doos...,1beonga-ui Gijeok (Miracle on 1st Street) (2007),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,40228.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
2,"1\n Come on man, let's get in the picture.\n B...",200 Degrees (2017),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,460711.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
3,- None of that ever happened.\n - Ever.\n Ther...,21 & Over (2013),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,107811.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Roberto!\n Roberto. Look, you can level with m...",4 mosche di velluto grigio (The Four Velvet Fl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19325,"2\n Ethan,\n How do you know\n that what you'r...",implanted (2013),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,871812.0,NaN,NaN,NaN,1.0,2.0,NaN,NaN,NaN,NaN
19326,You will be arrested.\n Don't worry. I'll come...,uwantme2killhim? (2013),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,202960.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN
19327,"He stole the chip, he's inside.\n l see him.\n...",xXx (2002),1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,7451.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
19328,You know how I came up with the idea\n for the...,xXx: Return of Xander Cage (2017),1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,47971.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
current_chunk = 0
chunk_size = 1200
chunks = np.array_split(df, len(df) // chunk_size)
nbchunks = len(chunks)
results = []

In [ ]:
df_final = pd.concat(results)
df_final
#df_final.to_csv("MovieData.csv", index=False)